# Freddie Mac Mortgage Risk Project
## Notebook 06: Model Interpretation and Policy Recommendations

This notebook interprets the challenger model, evaluates practical threshold choices, translates predicted risk into operational score bands, and develops a first-pass policy framework for underwriting review and portfolio monitoring. The objective is to convert model performance into actionable recommendations for a mortgage risk team.

## 1. Import packages and define project paths

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

from google.colab import drive
drive.mount('/content/drive')

from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

BASE = Path("/content/drive/MyDrive/freddie_mac_risk_project")
PROCESSED = BASE / "data" / "processed"
TABLES = BASE / "outputs" / "tables"

print("Processed path exists:", PROCESSED.exists())
print("Tables path exists:", TABLES.exists())

Mounted at /content/drive
Processed path exists: True
Tables path exists: True


## 2. Load the modeling base

In [2]:
modeling_base = pd.read_parquet(PROCESSED / "modeling_base_with_target_2019_2023.parquet")

print("Modeling base shape:", modeling_base.shape)
modeling_base.head()

Modeling base shape: (250000, 34)


,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,msa,mi_pct,num_units,occupancy_status,cltv,dti,...,servicer_name,super_conforming_flag,pre_harp_loan_sequence_number,program_indicator,relief_refinance_indicator,property_valuation_method,interest_only_indicator,mi_cancellation_indicator,vintage_year,target_12m_serious_dq
0,741,2019-03-01,N,2049-02-01,None,0,1,P,80,33,...,Other servicers,None,None,9,None,7,N,7,2019,0
1,731,2019-03-01,N,2049-02-01,13460,0,1,P,77,44,...,Other servicers,None,None,9,None,7,N,7,2019,0
2,722,2019-03-01,N,2049-02-01,None,30,1,P,95,41,...,Other servicers,None,None,9,None,7,N,N,2019,0
3,729,2019-03-01,N,2049-02-01,None,25,1,P,87,17,...,Other servicers,None,None,9,None,7,N,N,2019,0
4,773,2019-03-01,N,2049-02-01,33700,0,1,P,29,43,...,Other servicers,None,None,9,None,7,N,7,2019,0


## 3. Recreate the challenger modeling dataset

In [3]:
baseline_features = [
    "credit_score",
    "first_time_homebuyer_flag",
    "msa",
    "mi_pct",
    "num_units",
    "occupancy_status",
    "cltv",
    "dti",
    "original_upb",
    "ltv",
    "interest_rate",
    "channel",
    "property_state",
    "property_type",
    "postal_code",
    "loan_purpose",
    "original_loan_term",
    "num_borrowers",
    "seller_name",
    "servicer_name",
    "program_indicator",
    "property_valuation_method",
    "interest_only_indicator",
    "mi_cancellation_indicator",
    "vintage_year"
]

target_col = "target_12m_serious_dq"

baseline_df = modeling_base[baseline_features + [target_col]].copy()

train_df = baseline_df[baseline_df["vintage_year"].isin([2019, 2020, 2021, 2022])].copy()
test_df = baseline_df[baseline_df["vintage_year"] == 2023].copy()

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]

X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)
print("Train target rate:", y_train.mean())
print("Test target rate:", y_test.mean())

Train shape: (200000, 25) (200000,)
Test shape: (50000, 25) (50000,)
Train target rate: 0.011485
Test target rate: 0.00618


## 4. Prepare tree-model inputs

In [4]:
X_train_tree = X_train.copy()
X_test_tree = X_test.copy()

tree_numeric_features = [
    "credit_score",
    "mi_pct",
    "num_units",
    "cltv",
    "dti",
    "original_upb",
    "ltv",
    "interest_rate",
    "original_loan_term",
    "num_borrowers",
    "vintage_year"
]

tree_categorical_features = [col for col in X_train.columns if col not in tree_numeric_features]

for col in tree_numeric_features:
    X_train_tree[col] = pd.to_numeric(X_train_tree[col], errors="coerce")
    X_test_tree[col] = pd.to_numeric(X_test_tree[col], errors="coerce")

for col in tree_numeric_features:
    median_val = X_train_tree[col].median()
    X_train_tree[col] = X_train_tree[col].fillna(median_val)
    X_test_tree[col] = X_test_tree[col].fillna(median_val)

for col in tree_categorical_features:
    X_train_tree[col] = X_train_tree[col].fillna("Missing")
    X_test_tree[col] = X_test_tree[col].fillna("Missing")

ordinal_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
X_train_tree[tree_categorical_features] = ordinal_encoder.fit_transform(X_train_tree[tree_categorical_features])
X_test_tree[tree_categorical_features] = ordinal_encoder.transform(X_test_tree[tree_categorical_features])

print("Prepared challenger inputs successfully.")

Prepared challenger inputs successfully.


## 5. Retrain the challenger model

In [5]:
gbm_model = HistGradientBoostingClassifier(
    max_depth=6,
    learning_rate=0.05,
    max_iter=300,
    random_state=42
)

gbm_model.fit(X_train_tree, y_train)

gbm_test_proba = gbm_model.predict_proba(X_test_tree)[:, 1]

gbm_test_roc_auc = roc_auc_score(y_test, gbm_test_proba)
gbm_test_pr_auc = average_precision_score(y_test, gbm_test_proba)

print("GBM Test ROC-AUC:", gbm_test_roc_auc)
print("GBM Test PR-AUC:", gbm_test_pr_auc)

GBM Test ROC-AUC: 0.8056357870930376
GBM Test PR-AUC: 0.039339351729980473


## 6. Extract challenger feature importance

In [7]:
from sklearn.inspection import permutation_importance

perm_importance = permutation_importance(
    gbm_model,
    X_test_tree,
    y_test,
    n_repeats=5,
    random_state=42,
    scoring="average_precision"
)

feature_names = X_test_tree.columns.tolist()

feature_importance = pd.DataFrame({
    "feature": feature_names,
    "importance_mean": perm_importance.importances_mean,
    "importance_std": perm_importance.importances_std
}).sort_values("importance_mean", ascending=False).reset_index(drop=True)

feature_importance.head(15)

,feature,importance_mean,importance_std
0,credit_score,0.028764,0.001324
1,servicer_name,0.013485,0.001128
2,num_borrowers,0.012003,0.001630
3,seller_name,0.011618,0.001337
4,original_upb,0.011495,0.001566
5,cltv,0.006531,0.001710
6,msa,0.003478,0.000620
7,ltv,0.002386,0.001239
8,dti,0.001788,0.002384
9,property_state,0.001145,0.001276


Permutation importance shows that the challenger model relies on a mix of borrower-quality, leverage, loan-structure, and institution-related variables. Credit score emerges as the strongest feature, which is highly consistent with mortgage underwriting intuition. Variables such as CLTV, LTV, DTI, and original UPB also contribute meaningfully, supporting the view that leverage and affordability are important drivers of early delinquency risk. Some institution-related fields, such as seller and servicer identifiers, also rank highly; these may capture genuine operational or portfolio-composition differences, but they should be interpreted cautiously because they may be less stable than core borrower and loan characteristics.

## 7. Evaluate threshold behavior for the challenger model

In [8]:
thresholds = [0.01, 0.02, 0.05, 0.10, 0.20, 0.30, 0.50]

gbm_threshold_results = []

for threshold in thresholds:
    pred_label = (gbm_test_proba >= threshold).astype(int)

    gbm_threshold_results.append({
        "threshold": threshold,
        "flag_rate": pred_label.mean(),
        "precision": precision_score(y_test, pred_label, zero_division=0),
        "recall": recall_score(y_test, pred_label, zero_division=0),
        "f1": f1_score(y_test, pred_label, zero_division=0)
    })

gbm_threshold_results = pd.DataFrame(gbm_threshold_results)
gbm_threshold_results

,threshold,flag_rate,precision,recall,f1
0,0.01,0.19138,0.020065,0.621359,0.038874
1,0.02,0.07820,0.028389,0.359223,0.052619
2,0.05,0.01340,0.061194,0.132686,0.083759
3,0.10,0.00292,0.116438,0.055016,0.074725
4,0.20,0.00038,0.315789,0.019417,0.036585
5,0.30,0.00006,0.333333,0.003236,0.006410
6,0.50,0.00000,0.000000,0.000000,0.000000


The threshold table shows the expected tradeoff between review volume and bad-loan capture. Lower thresholds flag a larger share of the portfolio and recover more bad loans, while higher thresholds sharply reduce volume but also miss most delinquent cases. In this low-event mortgage setting, useful operating thresholds lie far below 0.50, which confirms that a conventional default classification threshold would be inappropriate for policy use. The table suggests that the practical threshold choice should depend on review-capacity constraints and desired capture levels rather than on a fixed machine-learning convention.

## 8. Create challenger-model risk bands

In [9]:
gbm_scored_test = X_test.copy()
gbm_scored_test["actual_bad"] = y_test.values
gbm_scored_test["score"] = gbm_test_proba

gbm_scored_test["risk_band"] = pd.qcut(
    gbm_scored_test["score"],
    q=5,
    labels=["Very Low", "Low", "Medium", "High", "Very High"]
)

gbm_band_summary = (
    gbm_scored_test.groupby("risk_band")
    .agg(
        loan_count=("actual_bad", "count"),
        bad_rate=("actual_bad", "mean"),
        avg_score=("score", "mean")
    )
    .reset_index()
)

gbm_band_summary

/tmp/ipykernel_154/1967427382.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  gbm_scored_test.groupby("risk_band")


,risk_band,loan_count,bad_rate,avg_score
0,Very Low,10000,0.0005,0.001127
1,Low,10000,0.0016,0.002076
2,Medium,10000,0.0021,0.003439
3,High,10000,0.0070,0.006521
4,Very High,10000,0.0197,0.023384


The score-band summary translates model probabilities into a more business-friendly risk segmentation framework. The most important validation check is whether bad rates rise consistently from the lowest band to the highest band. If they do, the model is producing a rank ordering that can plausibly support underwriting review priorities and portfolio monitoring decisions.

## 9. Attach provisional policy actions to the challenger score bands


In [10]:
def map_policy_action(band):
    band = str(band)
    if band == "Very Low":
        return "Auto-approve / standard monitoring"
    elif band == "Low":
        return "Approve / standard review"
    elif band == "Medium":
        return "Manual review / request additional documentation"
    elif band == "High":
        return "Senior underwriter review / tighter exception policy"
    else:
        return "Enhanced review / possible decline"

gbm_band_summary["policy_action"] = gbm_band_summary["risk_band"].map(map_policy_action)
gbm_band_summary

,risk_band,loan_count,bad_rate,avg_score,policy_action
0,Very Low,10000,0.0005,0.001127,Auto-approve / standard monitoring
1,Low,10000,0.0016,0.002076,Approve / standard review
2,Medium,10000,0.0021,0.003439,Manual review / request additional documentation
3,High,10000,0.0070,0.006521,Senior underwriter review / tighter exception ...
4,Very High,10000,0.0197,0.023384,Enhanced review / possible decline


## 10. Build bad-loan capture table for candidate thresholds

In [11]:
capture_table = []

total_bad_loans = y_test.sum()

for threshold in thresholds:
    pred_label = (gbm_test_proba >= threshold).astype(int)

    flagged_loans = pred_label.sum()
    captured_bad_loans = ((pred_label == 1) & (y_test == 1)).sum()

    capture_table.append({
        "threshold": threshold,
        "flagged_loans": int(flagged_loans),
        "flag_rate": pred_label.mean(),
        "captured_bad_loans": int(captured_bad_loans),
        "bad_loan_capture_rate": captured_bad_loans / total_bad_loans if total_bad_loans > 0 else 0
    })

capture_table = pd.DataFrame(capture_table)
capture_table

,threshold,flagged_loans,flag_rate,captured_bad_loans,bad_loan_capture_rate
0,0.01,9569,0.19138,192,0.621359
1,0.02,3910,0.07820,111,0.359223
2,0.05,670,0.01340,41,0.132686
3,0.10,146,0.00292,17,0.055016
4,0.20,19,0.00038,6,0.019417
5,0.30,3,0.00006,1,0.003236
6,0.50,0,0.00000,0,0.000000


The capture table translates model thresholds into an operational review framework by showing how many loans would be flagged and what share of bad loans would be captured at each cutoff. This is especially useful for a mortgage risk team because the threshold decision is fundamentally a capacity-allocation problem: lower cutoffs increase bad-loan capture but require reviewing a larger share of the portfolio, while higher cutoffs conserve review resources but miss more risk. The table therefore provides a practical bridge between predictive performance and underwriting workflow design.

## 11. Save interpretation and policy outputs

In [12]:
feature_importance.to_csv(TABLES / "gbm_feature_importance.csv", index=False)
gbm_threshold_results.to_csv(TABLES / "gbm_threshold_results.csv", index=False)
gbm_band_summary.to_csv(TABLES / "gbm_band_summary.csv", index=False)
capture_table.to_csv(TABLES / "gbm_capture_table.csv", index=False)

print(TABLES / "gbm_feature_importance.csv")
print(TABLES / "gbm_threshold_results.csv")
print(TABLES / "gbm_band_summary.csv")
print(TABLES / "gbm_capture_table.csv")

/content/drive/MyDrive/freddie_mac_risk_project/outputs/tables/gbm_feature_importance.csv
/content/drive/MyDrive/freddie_mac_risk_project/outputs/tables/gbm_threshold_results.csv
/content/drive/MyDrive/freddie_mac_risk_project/outputs/tables/gbm_band_summary.csv
/content/drive/MyDrive/freddie_mac_risk_project/outputs/tables/gbm_capture_table.csv


## Conclusion

The challenger model has now been interpreted in business terms through feature importance, threshold tradeoff analysis, risk-band segmentation, and bad-loan capture estimates. These outputs move the project beyond predictive benchmarking and toward a more practical underwriting decision-support framework that can be communicated to a mortgage risk team.